# Scraping AWOIAF Character Quotes
Produces `characters_quotes.csv` with columns: `name`, `ID`, `quote`

- One row per quote (a character with 10 quotes gets 10 rows).
- **`quote`**: The quote text, with any hyperlinked character names replaced by their canonical `ID` slug. Unlinked plain text is left verbatim (same logic as `scrape_character_bios.ipynb`).

Quotes live **on the main character page** under headings like
"Quotes by Arya" or "Quotes about Arya" — not on a separate subpage.
We find those sections and extract every `<blockquote class="templatequote">` inside them.

Requires `characters.csv` from `scrape_characters.ipynb` (or re-scrapes the list if not found).

In [15]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import os
from urllib.parse import quote, unquote
from concurrent.futures import ThreadPoolExecutor

## Setup

In [16]:
BASE   = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

session = requests.Session()
session.headers.update(headers)

## 1. Load or scrape the character list

In [17]:
CHARACTERS_CSV = '../csvs/characters.csv'

def scrape_character_list():
    list_page = session.get(BASE + PREFIX + 'List_of_characters', timeout=20)
    print(f'List page status: {list_page.status_code}')
    soup = BeautifulSoup(list_page.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    characters = []
    seen_ids = set()
    for li in content.find_all('li'):
        a = li.find('a')
        if not a:
            continue
        href = a.get('href', '')
        if not href.startswith(PREFIX):
            continue
        if 'redlink=1' in href or 'action=' in href:
            continue
        slug = href[len(PREFIX):].split('#', 1)[0]
        if not slug or slug.startswith('Special:'):
            continue
        slug = unquote(slug)
        if slug in seen_ids:
            continue
        seen_ids.add(slug)
        name = a.get_text(strip=True)
        if not name:
            continue
        characters.append({'name': name, 'ID': slug})
    return characters

if os.path.exists(CHARACTERS_CSV):
    char_df = pd.read_csv(CHARACTERS_CSV)
    print(f'Loaded {len(char_df)} characters from {CHARACTERS_CSV}')
else:
    print(f'{CHARACTERS_CSV} not found — scraping character list...')
    chars = scrape_character_list()
    char_df = pd.DataFrame(chars, columns=['name', 'ID'])
    char_df.to_csv(CHARACTERS_CSV, index=False)
    print(f'Saved {len(char_df)} characters to {CHARACTERS_CSV}')

characters = char_df.to_dict('records')
character_ids = set(row['ID'] for row in characters)
print(f'Character set: {len(character_ids)} IDs')

Loaded 3690 characters from characters.csv
Character set: 3690 IDs


## 2. Helpers

In [18]:
def slug_to_url(slug):
    """Build a full wiki URL for a slug."""
    return BASE + PREFIX + quote(slug, safe="_/(),'")


def href_to_slug(href):
    """Return the canonical (URL-decoded) slug for an internal wiki link, or None."""
    if not href.startswith(PREFIX):
        return None
    if 'redlink=1' in href or 'action=' in href:
        return None
    slug = href[len(PREFIX):].split('#', 1)[0]
    if not slug or slug.startswith('Special:'):
        return None
    return unquote(slug)


def node_to_text(node, character_ids):
    """
    Recursively convert a BeautifulSoup node to text.
    If a text node is inside an <a> that links to a known character, replace
    the display text with the target slug. Otherwise keep the raw text.
    This is identical in principle to para_to_text() in the bio scraper.
    """
    parts = []
    for leaf in node.descendants:
        if leaf.name is not None:
            continue  # skip tag nodes, only process text leaves
        text = str(leaf)
        if not text:
            continue
        parent = leaf.parent
        if parent and parent.name == 'a':
            href = parent.get('href', '')
            slug = href_to_slug(href)
            if slug and slug in character_ids:
                parts.append(slug)
                continue
        parts.append(text)
    return ''.join(parts)


def clean_quote(text):
    """Strip citation markers and normalise whitespace."""
    text = re.sub(r'\[\d+\]', '', text)       # [1], [2] ...
    text = re.sub(r'\[nb \d+\]', '', text)    # [nb 1] ...
    text = re.sub(r'\s+', ' ', text)           # collapse whitespace
    return text.strip()

## 3. Quote extraction

AWOIAF quote pages use the MediaWiki `templatequote` structure:

```html
<blockquote class="templatequote">
  <div class="templatequotetext">The quote text, with <a href="...">linked names</a>.</div>
  <div class="templatequotecite">— <a href="/index.php/Arya_Stark">Arya</a> to <a href="/index.php/Jon_Snow">Jon</a>, A Game of Thrones</div>
</blockquote>
```

From `templatequotetext` we get the quote text (with linked character names → IDs).
From `templatequotecite` we extract:
- `speaker_id`: the first linked character in the cite (the one speaking)
- `recipient_id`: the second linked character, if present (the one spoken to)

Both are `''` if no character link is found in that position.

In [19]:
def parse_cite(cite_div, character_ids):
    """
    Extract speaker and recipient IDs from a templatequotecite div.
    The cite typically looks like:
      '— Arya to Jon Snow, A Game of Thrones'
    where 'Arya' and 'Jon Snow' are <a> links to character pages.

    Returns (speaker_id, recipient_id) — either may be '' if not found.
    Speaker = first character link in the cite.
    Recipient = second character link in the cite.
    """
    if cite_div is None:
        return '', ''
    char_links = []
    for a in cite_div.find_all('a'):
        slug = href_to_slug(a.get('href', ''))
        if slug and slug in character_ids:
            char_links.append(slug)
    speaker_id   = char_links[0] if len(char_links) > 0 else ''
    recipient_id = char_links[1] if len(char_links) > 1 else ''
    return speaker_id, recipient_id


def extract_quotes(soup, character_ids):
    """
    Find all templatequote blockquotes on the page.
    Returns a list of dicts: {quote, speaker_id, recipient_id}.
    """
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return []

    results = []

    for bq in content.find_all('blockquote', class_='templatequote'):
        text_div = bq.find('div', class_='templatequotetext')
        if not text_div:
            continue
        text = clean_quote(node_to_text(text_div, character_ids))
        if not text:
            continue
        cite_div = bq.find('div', class_='templatequotecite')
        speaker_id, recipient_id = parse_cite(cite_div, character_ids)
        results.append({
            'quote':        text,
            'speaker_id':   speaker_id,
            'recipient_id': recipient_id,
        })

    return results

## 4. Per-character quote scrape

In [20]:
def scrape_quotes(row):
    """
    Fetch the main character page and extract all templatequote blockquotes.
    Quotes live on the main page under headings like 'Quotes by X' / 'Quotes about X'.
    We don't need to find those headings — just grab every <blockquote class='templatequote'>
    on the page directly.
    Returns a list of dicts: [{'name': ..., 'ID': ..., 'quote': ...}, ...]
    """
    name, slug = row['name'], row['ID']
    url = slug_to_url(slug)

    try:
        resp = session.get(url, timeout=20)
    except requests.RequestException as e:
        print(f'  ! request failed for {slug}: {e}')
        return []

    if resp.status_code != 200:
        print(f'  ! HTTP {resp.status_code} for {slug}')
        return []

    soup = BeautifulSoup(resp.content, 'html.parser')
    results = extract_quotes(soup, character_ids)

    return [
        {'name': name, 'ID': slug, 'quote': r['quote'],
         'speaker_id': r['speaker_id'], 'recipient_id': r['recipient_id']}
        for r in results
    ]

## 5. Run the scrape
Parallelised across 8 threads; checkpoints every 100 characters processed.

In [21]:
OUT     = '../csvs/characters_quotes.csv'
COLUMNS = ['name', 'ID', 'quote', 'speaker_id', 'recipient_id']
WORKERS = 8

all_rows = []
chars_with_quotes = 0

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    for i, quote_rows in enumerate(executor.map(scrape_quotes, characters), 1):
        if quote_rows:
            all_rows.extend(quote_rows)
            chars_with_quotes += 1
        if i % 50 == 0:
            print(f'  {i}/{len(characters)} characters checked — {chars_with_quotes} with quotes, {len(all_rows)} total quotes so far')
        if i % 100 == 0:
            pd.DataFrame(all_rows, columns=COLUMNS).to_csv(OUT, index=False)

pd.DataFrame(all_rows, columns=COLUMNS).to_csv(OUT, index=False)
print(f'\nDone.')
print(f'  Characters with quotes : {chars_with_quotes}')
print(f'  Total quotes collected : {len(all_rows)}')
print(f'  Saved to               : {OUT}')

  50/3690 characters checked — 23 with quotes, 122 total quotes so far
  100/3690 characters checked — 41 with quotes, 202 total quotes so far
  150/3690 characters checked — 56 with quotes, 254 total quotes so far
  200/3690 characters checked — 72 with quotes, 320 total quotes so far
  250/3690 characters checked — 86 with quotes, 389 total quotes so far
  300/3690 characters checked — 106 with quotes, 499 total quotes so far
  350/3690 characters checked — 124 with quotes, 569 total quotes so far
  400/3690 characters checked — 140 with quotes, 620 total quotes so far
  450/3690 characters checked — 153 with quotes, 653 total quotes so far
  500/3690 characters checked — 167 with quotes, 717 total quotes so far
  550/3690 characters checked — 177 with quotes, 779 total quotes so far
  600/3690 characters checked — 189 with quotes, 818 total quotes so far
  650/3690 characters checked — 206 with quotes, 890 total quotes so far
  700/3690 characters checked — 228 with quotes, 976 tota

## 6. Sanity check

In [22]:
df = pd.read_csv(OUT)
print(f'Shape: {df.shape}')
print(f'Unique characters with quotes : {df["ID"].nunique()}')
print(f'Quotes with speaker_id        : {df["speaker_id"].ne("").sum()}')
print(f'Quotes with recipient_id      : {df["recipient_id"].ne("").sum()}')
print()

# Characters with the most quotes
print('Top 10 characters by quote count:')
print(df.groupby(['name', 'ID']).size().sort_values(ascending=False).head(10))
print()

# Sample quotes for a well-known character
sample = df[df['name'] == 'Tyrion Lannister']
if not sample.empty:
    print(f'--- Tyrion Lannister ({len(sample)} quotes) — first 3 ---')
    for _, row in sample.head(3).iterrows():
        print(f'  quote      : "{row["quote"][:100]}"')
        print(f'  speaker_id : {row["speaker_id"]}')
        print(f'  recipient  : {row["recipient_id"]}')
        print()

Shape: (4626, 5)
Unique characters with quotes : 1167
Quotes with speaker_id        : 4626
Quotes with recipient_id      : 4626

Top 10 characters by quote count:
name                ID                
Theon Greyjoy       Theon_Greyjoy         28
Sandor Clegane      Sandor_Clegane        25
Jaime Lannister     Jaime_Lannister       24
Tywin Lannister     Tywin_Lannister       23
Stannis Baratheon   Stannis_Baratheon     22
Petyr Baelish       Petyr_Baelish         22
Eddard Stark        Eddard_Stark          22
Robert I Baratheon  Robert_I_Baratheon    21
Sansa Stark         Sansa_Stark           21
Jeyne Poole         Jeyne_Poole           21
dtype: int64

--- Tyrion Lannister (14 quotes) — first 3 ---
  quote      : "Tyrion: Let me give you some counsel, bastard. Never forget what you are, for surely the world will "
  speaker_id : Jon_Snow
  recipient  : nan

  quote      : "My mind is my weapon. Jaime_Lannister has his sword, King Robert_I_Baratheon has his warhammer, and "
  speak

---
## Notes

### Output format
One row per quote. A character with 20 quotes has 20 rows. Join on `ID` to merge with `characters_bio.csv` or `characters_enriched.csv`.

### Character name → ID substitution
Same link-based logic as the bio scraper: only hyperlinked names are replaced with their target slug. Plain text is left verbatim. This means:
- `"Eddard_Stark is a man of honour"` — Ned was linked on the page
- `"my father was a man of honour"` — unlinked, kept as-is

### Downstream uses
- **Sentiment / relationship extraction**: who does each character mention, and in what tone?
- **Stylometry**: do characters have distinct speech patterns? (sentence length, vocabulary, formality)
- **Quote-based network**: build edges from co-mentions within a single quote (strong signal — if A's quote mentions B, that's a direct relationship)